# Clasificación con KNN, Perceptrón y Naive Bayes

En este notebook se entrenan **tres modelos por separado** usando el dataset de pérdidas de equipamiento de Rusia.

In [ ]:
import pandas as pd

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import Perceptron
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [2]:
# Carga de datos
csv_path = Path('russian_ukraine_war/russia_losses_equipment.csv')
df = pd.read_csv(csv_path)

# Limpieza básica
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df = df.sort_values('day').reset_index(drop=True)

# Convertimos columnas numéricas (excepto fecha y texto)
ignore_cols = ['date', 'greatest losses direction']
feature_candidates = [col for col in df.columns if col not in ignore_cols]
for col in feature_candidates:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df[feature_candidates] = df[feature_candidates].fillna(0)

# El dataset es acumulado, así que calculamos variación diaria
daily = df[feature_candidates].diff().fillna(0)
daily['day'] = df['day']
daily.head()

,day,aircraft,helicopter,tank,APC,field artillery,MRL,military auto,fuel tank,drone,naval ship,anti-aircraft warfare,special equipment,mobile SRBM system,vehicles and fuel tanks,cruise missiles,submarines
0,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,3,17.0,19.0,66.0,190.0,0.0,0.0,30.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,4,0.0,0.0,4.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,5,2.0,3.0,0.0,110.0,24.0,17.0,161.0,0.0,1.0,0.0,5.0,0.0,0.0,0.0,0.0,0.0
4,6,0.0,0.0,48.0,30.0,3.0,3.0,14.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0


In [4]:
# Variable objetivo binaria:
# 1 si la pérdida diaria de drones está por encima de la mediana, 0 en caso contrario
target_col = 'drone'
threshold = daily[target_col].median()
y = (daily[target_col] > threshold).astype(int)

# Features: todas menos la variable objetivo
X = daily.drop(columns=[target_col])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('Shape X_train:', X_train.shape)
print('Shape X_test:', X_test.shape)
print('Distribución y (0/1):')
print(y.value_counts(normalize=True).rename('proportion'))

Shape X_train: (1172, 16)
Shape X_test: (294, 16)
Distribución y (0/1):
drone
0    0.502046
1    0.497954
Name: proportion, dtype: float64


## 1) K-Nearest Neighbors (KNN)

In [5]:
knn_model = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier(n_neighbors=7))
])

knn_model.fit(X_train, y_train)
knn_pred = knn_model.predict(X_test)

print('KNN Accuracy:', round(accuracy_score(y_test, knn_pred), 4))
print('KNN Confusion Matrix:\n', confusion_matrix(y_test, knn_pred))
print('KNN Classification Report:\n', classification_report(y_test, knn_pred))

KNN Accuracy: 0.8741
KNN Confusion Matrix:
 [[138  10]
 [ 27 119]]
KNN Classification Report:
               precision    recall  f1-score   support

           0       0.84      0.93      0.88       148
           1       0.92      0.82      0.87       146

    accuracy                           0.87       294
   macro avg       0.88      0.87      0.87       294
weighted avg       0.88      0.87      0.87       294



## 2) Perceptrón

In [6]:
perceptron_model = Pipeline([
    ('scaler', StandardScaler()),
    ('perceptron', Perceptron(random_state=42, max_iter=2000, tol=1e-3))
])

perceptron_model.fit(X_train, y_train)
perceptron_pred = perceptron_model.predict(X_test)

print('Perceptrón Accuracy:', round(accuracy_score(y_test, perceptron_pred), 4))
print('Perceptrón Confusion Matrix:\n', confusion_matrix(y_test, perceptron_pred))
print('Perceptrón Classification Report:\n', classification_report(y_test, perceptron_pred))

Perceptrón Accuracy: 0.8639
Perceptrón Confusion Matrix:
 [[131  17]
 [ 23 123]]
Perceptrón Classification Report:
               precision    recall  f1-score   support

           0       0.85      0.89      0.87       148
           1       0.88      0.84      0.86       146

    accuracy                           0.86       294
   macro avg       0.86      0.86      0.86       294
weighted avg       0.86      0.86      0.86       294



## 3) Naive Bayes (GaussianNB)

In [7]:
nb_model = GaussianNB()

nb_model.fit(X_train, y_train)
nb_pred = nb_model.predict(X_test)

print('Naive Bayes Accuracy:', round(accuracy_score(y_test, nb_pred), 4))
print('Naive Bayes Confusion Matrix:\n', confusion_matrix(y_test, nb_pred))
print('Naive Bayes Classification Report:\n', classification_report(y_test, nb_pred))

Naive Bayes Accuracy: 0.5782
Naive Bayes Confusion Matrix:
 [[ 26 122]
 [  2 144]]
Naive Bayes Classification Report:
               precision    recall  f1-score   support

           0       0.93      0.18      0.30       148
           1       0.54      0.99      0.70       146

    accuracy                           0.58       294
   macro avg       0.73      0.58      0.50       294
weighted avg       0.74      0.58      0.50       294

